# L4a: Kernel Functions and Kernel Regression
In this lecture, we'll take a theoretical detour into positive-definite kernels and explore how they power our first kernel machine: kernel regression. You'll discover how similarity functions enable us to solve nonlinear problems using elegant linear methods.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Kernel Function Definition:__ Understand what kernel functions are, their role as similarity measures, and the mathematical requirements (symmetry, positive semi-definiteness) for valid kernels.
> * __Gram Matrix and Positive Semi-Definiteness:__ Learn how to construct Gram matrices from data and verify that kernel matrices satisfy positive semi-definiteness constraints.
> * __Kernel Ridge Regression:__ Understand how kernel functions enable non-parametric regression through the "kernel trick" and how to derive the dual formulation with α coefficients.


Let's get started!
___

## Example
Today, we will use the following examples to illustrate key concepts:
 
> [▶ Can we estimate the similarity of different firms?](CHEME-5820-L4a-Example-MeasureFirmSimilarityScores-Spring-2026.ipynb). In this example, let's explore how to measure the similarity between different firms based upon the similarity of their daily growth rates over 10-year periods. Does this similarity correlate with other firm metrics, e.g., business sector, market capitalization, etc.?
___

## Kernel Functions
Kernel functions let algorithms operate in high-dimensional spaces without explicitly building the coordinates.

> __What are kernel functions__?
>
> A kernel function $k:\mathbb{R}^{m}\times\mathbb{R}^{m}\to\mathbb{R}$ maps two vectors to a scalar similarity score. Valid kernels are symmetric and positive semi-definite.
> 
> Common kernels include the linear kernel $k(\mathbf{z}_i, \mathbf{z}_j) = \mathbf{z}_i^{\top}\mathbf{z}_j$, the polynomial kernel $k_{d}(\mathbf{z}_i, \mathbf{z}_j) = (1+\mathbf{z}_i^{\top}\mathbf{z}_j)^d$, and the RBF kernel $k_{\gamma}(\mathbf{z}_i, \mathbf{z}_j) = \exp\left(-\gamma \left\|\mathbf{z}_i - \mathbf{z}_j\right\|_{2}^{2}\right)$ with $\gamma>0$.



__What do kernel functions do__?

Kernels measure similarity between points in some feature space. We can use them directly, or use them to represent implicit feature engineering (e.g., adding $x^{2}, x^{3}, \dots$) without explicitly constructing those features.

Let's look at example kernels from [the `KernelFunctions.jl` package](https://github.com/JuliaGaussianProcesses/KernelFunctions.jl) and apply them to a financial dataset.

> __Example__
>
> If two tickers are in similar industries (e.g., `NVDA` and `AMD`, or `GS` and `JPM`), their similarity score should be high. Anticorrelated tickers (e.g., `SPY` or `QQQ` and `GLD`) should have low similarity scores.
>
> [▶ Can we estimate the similarity of different firms?](CHEME-5820-L4a-Example-MeasureFirmSimilarityScores-Spring-2026.ipynb)

### Kernels correspond to inner products in feature spaces

Every positive-definite kernel corresponds to an inner product in some feature space. By Mercer's theorem, there exists a transformation $\phi$ such that:

$$k(\mathbf{x},\mathbf{z}) = \langle \phi(\mathbf{x}), \phi(\mathbf{z}) \rangle$$

So kernels compute inner products in a transformed space $\mathcal{H}$ without explicitly constructing that space. In that space, the model is linear:

$$
\boxed{
    \hat y(\mathbf{z})
    = \phi(\mathbf{z})^\top\,\hat\theta_\lambda
    = \sum_{i=1}^n \alpha_i\,\underbrace{\langle \phi(\mathbf{z}),\,\phi(\mathbf{x}_i)\rangle}_{\text{similarity in $\mathcal H$}}
    = \sum_{i=1}^n \alpha_i\,k(\mathbf{z},\mathbf{x}_i).}
$$

**Concrete example: A simple polynomial kernel**

For scalars $v_i, v_j \in \mathbb{R}$, consider:

$$k(v_i, v_j) = (1 + v_i v_j)^2$$

Expand:
$$k(v_i, v_j) = 1 + 2v_i v_j + (v_i v_j)^2$$

Define $\phi(v) = [1, \sqrt{2}v, v^2]^{\top}$. Then:

$$\langle \phi(v_i), \phi(v_j) \rangle = 1 + 2v_i v_j + (v_i v_j)^2 \quad\blacksquare$$

**Takeaway:** The quadratic kernel maps 1D input into a 3D feature space. Nonlinearity in the original space becomes a linear inner product in the transformed space.

___

### Can any function be a kernel function?

Not every function is a valid kernel. Valid kernels must satisfy strict mathematical constraints.

> __Rules for a valid kernel function__:
>
> A function $k:\mathbb{R}^{m}\times\mathbb{R}^{m}\to\mathbb{R}$ is valid if, for any finite set $\{\mathbf{v}_1, \dots, \mathbf{v}_n\}$, the kernel matrix $\mathbf{K}$ with entries $K_{ij} = k(\mathbf{v}_i, \mathbf{v}_j)$ is symmetric and positive semidefinite. Equivalently, all eigenvalues of $\mathbf{K}$ are non-negative and $\mathbf{x}^{\top}\mathbf{K}\mathbf{x} \geq 0$ for any real vector $\mathbf{x}$.

__The Gram matrix as a special case__: A Gram matrix uses the inner product kernel. For vectors $\mathbf{v}_1,\dots,\mathbf{v}_n \in \mathbb{R}^{m}$:

$$K_{ij} = k(\mathbf{v}_i, \mathbf{v}_j) = \langle\mathbf{v}_{i},\mathbf{v}_{j}\rangle$$

If the vectors are columns of $\mathbf{X} \in \mathbb{R}^{m \times n}$, then $\mathbf{K} = \mathbf{X}^{\top}\mathbf{X}$. If they are rows of $\mathbf{X} \in \mathbb{R}^{n \times m}$, then $\mathbf{K} = \mathbf{X}\mathbf{X}^{\top}$. In both cases, $\mathbf{K}$ is symmetric and positive semidefinite.

Next, we will compute a Gram matrix from our financial data and examine its eigendecomposition.

___

## Kernel regression

Now that we know what makes a valid kernel, we can use kernels for nonlinear regression.

Kernel regression is non-parametric: it predicts outputs using weighted combinations of training examples rather than a single global model.

> __How does it work__? Kernel regression assigns weights to training points based on similarity to a query point. The prediction is a weighted sum of those points.

Suppose we have a dataset $\mathcal{D} = \{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\}$ with features $\mathbf{x}_i \in \mathbb{R}^{m}$ and targets $y_i \in \mathbb{R}$. We can model this as linear regression:
$$
\hat{\mathbf{y}} = \hat{\mathbf{X}}\theta
$$
where $\hat{\mathbf{X}}$ contains augmented feature vectors and $\theta\in\mathbb{R}^{p}$ with $p=m+1$. The ridge solution is:
$$
\hat{\mathbf{\theta}}_{\lambda} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
$$

#### Kernel ridge regression
Write the parameter vector as a weighted sum of training examples: $\hat{\theta}_{\lambda} = \sum_{i=1}^{n}\alpha_{i}\hat{\mathbf{x}}_{i}$. This leads to predictions based on inner products only.

> __Why this representation matters__: It lets us replace inner products with kernel evaluations, enabling implicit feature engineering.

For a new point $\hat{\mathbf{z}}$:
$$
\begin{align*}
\hat{y} & = \hat{\mathbf{z}}^{\top}\theta = \sum_{i=1}^{n}\alpha_{i}\left\langle\hat{\mathbf{z}},\hat{\mathbf{x}}_{i}\right\rangle\\
        & = \sum_{i=1}^{n}\alpha_{i}\,k(\hat{\mathbf{z}},\hat{\mathbf{x}}_{i})
\end{align*}
$$

To solve for $\alpha_i$, we need the matrix of all pairwise inner products between training points. Because each training example appears as a row of $\hat{\mathbf{X}}$, that matrix is $\mathbf{K} = \hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}$. This is the Gram matrix in sample space, and it is symmetric and positive semi-definite by construction.

The two expressions for $\hat{\theta}_{\lambda}$ can be equated. Starting from $\hat{\theta}_{\lambda} = \hat{\mathbf{X}}^{\top}\alpha$, we substitute into the ridge regression optimality condition $(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I})\hat{\theta}_{\lambda} = \hat{\mathbf{X}}^{\top}\mathbf{y}$:
$$
\begin{align*}
(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I})\hat{\mathbf{X}}^{\top}\alpha &= \hat{\mathbf{X}}^{\top}\mathbf{y}\\
\hat{\mathbf{X}}(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda\,\mathbf{I})\hat{\mathbf{X}}^{\top}\alpha & = \hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\mathbf{y}\quad\mid\text{multiply both sides by $\hat{\mathbf{X}}$ on the left} \\
(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}+\lambda\,\mathbf{I})\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\alpha & = \hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\mathbf{y}\quad\mid\text{key step: see explanation below}\\
(\mathbf{K}+\lambda\,\mathbf{I})\mathbf{K}\alpha & = \mathbf{K}\mathbf{y}\quad\mid\text{substitute $\mathbf{K} = \hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}$}\\
\alpha &= (\mathbf{K}+\lambda\mathbf{I})^{-1}\mathbf{K}\mathbf{y}\quad\mid\text{multiply both sides by $(\mathbf{K}+\lambda\mathbf{I})^{-1}$ on the left}
\end{align*}
$$

> __Understanding the key transformation step__:
>
> The identity
> $$\hat{\mathbf{X}}(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\lambda \mathbf{I})\hat{\mathbf{X}}^{\top} = (\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}+\lambda \mathbf{I})\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}$$
> follows by expanding both sides. It moves the computation from feature space ($p\times p$) to sample space ($n\times n$), which is cheaper when $p$ is large.

where $\mathbf{K} = \hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}$ is the Gram matrix of the augmented features (which is symmetric and positive semi-definite, as guaranteed by the kernel validity properties learned above), the matrix $\mathbf{I}$ denotes the identity matrix, the vector $\mathbf{y}$ denotes the observed outputs and $\lambda\geq{0}$ denotes the regularization parameter.

In practice, $\alpha = (\mathbf{K}+\lambda\mathbf{I})^{-1}\mathbf{K}\mathbf{y}$ simplifies in implementation since we can factor or directly solve the linear system.

> __Technical Note__:
>
> We derived $\alpha$ using the linear kernel Gram matrix $\mathbf{K} = \hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}$. The solution depends only on kernel evaluations, so any valid kernel can replace $\mathbf{K}$ without changing the algorithm. The Woodbury identity explains why the sample-space system $(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}+\lambda\mathbf{I})^{-1}$ is computationally favorable when feature dimension is large.

#### The Kernel Trick

The key insight of kernel methods is that predictions depend only on inner products between data points. By substituting $k(\hat{\mathbf{z}},\hat{\mathbf{x}}_{i})$ for $\left\langle\hat{\mathbf{z}},\hat{\mathbf{x}}_{i}\right\rangle$, we unlock the ability to work in high-dimensional feature spaces without explicitly constructing them.

> __The Kernel Trick__:
>
> Express predictions as inner products, then replace those inner products with kernel evaluations. The algorithm structure stays the same; only the similarity function changes.

This approach has profound implications: the algorithm never changes. We always solve $\alpha = (\mathbf{K}+\lambda\mathbf{I})^{-1}\mathbf{K}\mathbf{y}$ using a kernel matrix. By choosing different kernels (linear, polynomial, RBF), we automatically adapt to different feature spaces while maintaining the same mathematical framework.

___

## Summary
Kernel functions are similarity measures that enable non-parametric modeling of complex relationships without explicitly constructing high-dimensional feature representations.

> __Key Takeaways:__
>
> * __Valid kernels must be symmetric and positive semi-definite__, ensuring the corresponding Gram matrices satisfy important mathematical properties for all possible data choices.
> * __The kernel trick__ allows us to work implicitly in high-dimensional spaces by replacing inner products $\langle\mathbf{z}_i, \mathbf{z}_j\rangle$ with kernel evaluations $k(\mathbf{z}_i, \mathbf{z}_j)$, enabling scalable algorithms.
> * __Kernel ridge regression__ reformulates linear regression using a dual representation with coefficients α, shifting from a model-centric view (fitting θ) to a data-centric view (weighted training examples).


Kernel methods form the mathematical foundation for support vector machines (SVMs), Gaussian processes, and other modern machine learning algorithms.

___